# 🏔️ OE Data Intelligence — Medallion Pipeline

**Full end-to-end pipeline from raw sys logs → Gold KPI tables**

| Source | Log Format | Issues Simulated |
|---|---|---|
| **Aternity** | JSONL | Teams/Outlook degradation, high response times |
| **NetScout** | CSV | WAN packet loss, congestion, hardware faults |
| **Intune** | JSONL | Non-compliant devices, Win7/8, MDM check-in failures |
| **Infoblox** | Syslog | DNS latency, NXDOMAIN, suspicious queries |
| **Salesforce** | CSV | P1/P2 incidents, SLA breaches, recurring issues |
| **ScienceLogic** | JSONL | Infrastructure alerts, threshold breaches |

## Step 1 — Generate Mock Log Files

In [1]:
import subprocess
result = subprocess.run(
    ['python', '/home/jovyan/scripts/generate_mock_logs.py', '--days', '3'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


🔄 Generating 3 day(s) of mock logs...

  ✅ Aternity: 7,500 records
  ✅ NetScout: 9,000 records
  ✅ Intune: 1,200 records
  ✅ Infoblox: 24,000 records
  ✅ Salesforce: 20 records
  ✅ ScienceLogic: 4,192 records

✅ Done! 6 log files written to mock-logs/
   Run the ingestion pipeline next:
   python pipelines/run_pipeline.py



## Step 2 — Preview Raw Logs

In [2]:
import json
from pathlib import Path

# Preview Aternity log
print('=== Aternity (JSONL sample) ===')
lines = Path('/home/jovyan/mock-logs/aternity').glob('*.jsonl')
for f in lines:
    with open(f) as fp:
        for i, line in enumerate(fp):
            if i < 3:
                print(json.dumps(json.loads(line), indent=2))
    break

=== Aternity (JSONL sample) ===
{
  "timestamp": "2026-05-09T23:08:12.491Z",
  "app_name": "Salesforce CRM",
  "user_id": "Fort Bragg NC",
  "user_email": "Fort Bragg NC",
  "device_id": "DEV-003",
  "device_name": "MOBILE-SGT-01",
  "session_id": "SES-468923",
  "response_time_ms": 892,
  "page_load_ms": 612,
  "experience_score": 59.5,
  "crash_flag": false,
  "error_code": "NONE",
  "client_os": "iOS 17.4",
  "location": "sgt.johnson@mail.mil",
  "network_segment": "DISA-WAN-BRAGG",
  "data_source": "ATERNITY"
}
{
  "timestamp": "2026-05-09T23:08:28.491Z",
  "app_name": "myPay (DFAS)",
  "user_id": "Fort Bragg NC",
  "user_email": "Fort Bragg NC",
  "device_id": "DEV-003",
  "device_name": "MOBILE-SGT-01",
  "session_id": "SES-719967",
  "response_time_ms": 354,
  "page_load_ms": 211,
  "experience_score": 79.7,
  "crash_flag": false,
  "error_code": "NONE",
  "client_os": "iOS 17.4",
  "location": "sgt.johnson@mail.mil",
  "network_segment": "DISA-LAN-ARLINGTON-02",
  "data_source"

In [3]:
# Preview Infoblox syslog
print('=== Infoblox DNS Syslog (first 5 lines) ===')
logs = list(Path('/home/jovyan/mock-logs/infoblox').glob('*.log'))
if logs:
    with open(logs[0]) as f:
        for i, line in enumerate(f):
            if i < 5: print(line.strip())

=== Infoblox DNS Syslog (first 5 lines) ===
May 09 23:08:00 dns-secondary-arlington.disa.mil named[1234]: client 10.158.54.142#53: query: disa.mil IN A response: NOERROR (15.54ms)
May 09 23:08:29 dns-backup-belvoir.disa.mil named[1234]: client 10.143.240.88#53: query: nonexistent.mil IN PTR response: NXDOMAIN (12.23ms)
May 09 23:08:34 dns-primary-meade.disa.mil named[1234]: client 10.9.169.55#53: query: s3.us-gov-west-1.amazonaws.com IN CNAME response: NOERROR (321.08ms)
May 09 23:08:41 dns-secondary-arlington.disa.mil named[1234]: client 10.199.192.26#53: query: teams.microsoft.com IN PTR response: NOERROR (18.62ms)
May 09 23:08:45 dns-cloud-aws-east.disa.mil named[1234]: client 10.84.182.94#53: query: s3.us-gov-west-1.amazonaws.com IN CNAME response: NOERROR (2.61ms)


## Step 3 — Initialize Spark + Run Full Pipeline

In [6]:
import os
os.environ["SPARK_MASTER"] = "local[2]"  # force before any import

import sys
sys.path.insert(0, '/home/jovyan')

from spark_session import get_spark, Paths
spark = get_spark('OE-Pipeline-Notebook')
print("Spark ready:", spark.version)
print("Master:", spark.sparkContext.master)
# MUST print: local[2]

Spark ready: 3.5.0
Master: local[2]


In [7]:
from pipelines.run_pipeline import run
run()

  OE DATA INTELLIGENCE PLATFORM — FULL MEDALLION PIPELINE
  Sources: Aternity · NetScout · Intune · Infoblox ·
           Salesforce · ScienceLogic

──────────────────────────────────────────────────
  🥉  BRONZE — Raw Log Ingestion
──────────────────────────────────────────────────

📥 BRONZE: Aternity (App Performance)...
   ✅ 2,500 rows → s3a://bronze/raw_aternity

📥 BRONZE: NetScout (Network Flows)...
   ✅ 3,000 rows → s3a://bronze/raw_netscout

📥 BRONZE: Intune (Device Management)...
   ✅ 400 rows → s3a://bronze/raw_intune

📥 BRONZE: Infoblox (DNS Logs)...
   ✅ 8,000 rows → s3a://bronze/raw_infoblox

📥 BRONZE: Salesforce (Cases/Tickets)...
   ✅ 20 rows → s3a://bronze/raw_salesforce

📥 BRONZE: ScienceLogic (Infrastructure Alerts)...
   ✅ 1,397 rows → s3a://bronze/raw_sciencelogic

  ⏱  Bronze complete in 23.4s

──────────────────────────────────────────────────
  🥈  SILVER — Cleanse & Transform
──────────────────────────────────────────────────

🔧 SILVER: App Performance (Aternity)..

## Step 4 — Query Gold Tables (what the API serves)

In [9]:
# Reconnect spark session for queries
from spark_session import get_spark, Paths
spark = get_spark('OE-Query')

In [10]:
# App Health — show degraded/critical apps
print('=== App Health Summary (worst first) ===')
spark.read.format('delta').load(Paths.gold('app_health_summary')) \
    .select('app_name','health_status','experience_score',
            'avg_response_time_ms','crash_rate','active_user_count') \
    .orderBy('experience_score') \
    .show(10, truncate=False)

=== App Health Summary (worst first) ===
+---------------------+-------------+----------------+--------------------+----------+-----------------+
|app_name             |health_status|experience_score|avg_response_time_ms|crash_rate|active_user_count|
+---------------------+-------------+----------------+--------------------+----------+-----------------+
|Microsoft Teams      |CRITICAL     |38.4            |2723.0              |4.02      |5                |
|Outlook Web App      |CRITICAL     |44.5            |3082.0              |3.06      |5                |
|ServiceNow           |DEGRADED     |55.0            |1413.0              |0.5       |5                |
|Salesforce CRM       |DEGRADED     |60.0            |1120.0              |0.96      |5                |
|SAP ERP              |DEGRADED     |62.7            |1002.0              |2.29      |5                |
|NIPR Email           |HEALTHY      |72.5            |498.0               |0.51      |5                |
|SharePoint On

In [12]:
# Network Root Cause Analysis
print('=== Packet Loss Root Cause Analysis ===')
spark.read.format('delta').load(Paths.gold('packet_loss_root_cause')) \
    .select('segment_name','root_cause','packet_loss_rate',
            'confidence_score','severity','recommendation') \
    .orderBy('packet_loss_rate', ascending=False) \
    .show(truncate=False)

=== Packet Loss Root Cause Analysis ===
+---------------------+--------------------+----------------+----------------+--------+-----------------------------------------------------+
|segment_name         |root_cause          |packet_loss_rate|confidence_score|severity|recommendation                                       |
+---------------------+--------------------+----------------+----------------+--------+-----------------------------------------------------+
|DISA-WAN-DC-01       |CONGESTION          |3.199           |91.0            |CRITICAL|Implement QoS policy and consider bandwidth upgrade  |
|DISA-WAN-PENTAGON    |HARDWARE_DEGRADATION|2.73            |84.0            |CRITICAL|Schedule hardware replacement — error counters rising|
|DISA-WAN-MEADE-01    |LINK_SATURATION     |2.517           |84.0            |CRITICAL|Activate backup circuit or implement traffic shaping |
|DISA-LAN-ARLINGTON-01|DUPLEX_MISMATCH     |1.863           |76.0            |HIGH    |Force 1000/Full duple

In [13]:
# Device Compliance
print('=== Non-Compliant Devices ===')
spark.read.format('delta').load(Paths.gold('device_health_summary')) \
    .filter('compliance_state = "NON_COMPLIANT"') \
    .select('device_name','os_name','os_version','compliance_state',
            'days_since_checkin','assigned_user','location') \
    .orderBy('days_since_checkin', ascending=False) \
    .show(10, truncate=False)

=== Non-Compliant Devices ===
+--------------+-------+----------+----------------+------------------+--------------------+---------------+
|device_name   |os_name|os_version|compliance_state|days_since_checkin|assigned_user       |location       |
+--------------+-------+----------+----------------+------------------+--------------------+---------------+
|DESKTOP-OLD-01|Windows|7 SP1     |NON_COMPLIANT   |60                |legacy.user@mail.mil|Fort Belvoir VA|
|LAPTOP-OPS-01 |Windows|8.1       |NON_COMPLIANT   |60                |ops.analyst@mail.mil|Pentagon DC    |
|LAPTOP-CIO-01 |Windows|11 23H2   |NON_COMPLIANT   |60                |cio.staff@disa.mil  |Arlington VA   |
|TABLET-LT-01  |Android|13        |NON_COMPLIANT   |58                |lt.williams@mail.mil|Pentagon DC    |
|DESKTOP-OLD-01|Windows|7 SP1     |NON_COMPLIANT   |58                |legacy.user@mail.mil|Fort Belvoir VA|
|LAPTOP-CIV-01 |Windows|11 23H2   |NON_COMPLIANT   |58                |m.jones@mail.mil    |Arling

In [14]:
# DNS Issues
print('=== DNS Server Metrics ===')
spark.read.format('delta').load(Paths.gold('dns_metrics')) \
    .select('server','total_queries','failed_queries',
            'failure_rate','avg_response_ms','nxdomain_count') \
    .orderBy('avg_response_ms', ascending=False) \
    .show(truncate=False)

=== DNS Server Metrics ===
+--------------------------------+-------------+--------------+------------+---------------+--------------+
|server                          |total_queries|failed_queries|failure_rate|avg_response_ms|nxdomain_count|
+--------------------------------+-------------+--------------+------------+---------------+--------------+
|dns-primary-meade.disa.mil      |2081         |534           |25.66       |419.18         |298           |
|dns-backup-belvoir.disa.mil     |1950         |351           |18.0        |8.9            |256           |
|dns-cloud-aws-east.disa.mil     |1940         |340           |17.53       |8.81           |257           |
|dns-secondary-arlington.disa.mil|2029         |360           |17.74       |8.8            |266           |
+--------------------------------+-------------+--------------+------------+---------------+--------------+



In [15]:
# Open P1/P2 Issues
print('=== Critical Open Issues ===')
spark.read.format('delta').load(Paths.gold('top_issues_summary')) \
    .filter('severity IN ("P1","P2") AND status != "RESOLVED"') \
    .select('issue_id','severity','title','category','assigned_team') \
    .orderBy('severity') \
    .show(truncate=False)

=== Critical Open Issues ===
+--------------+--------+--------------------------------------------------+-----------+------------------+
|issue_id      |severity|title                                             |category   |assigned_team     |
+--------------+--------+--------------------------------------------------+-----------+------------------+
|INC-2024-08800|P1      |Microsoft Teams voice/video failure on WAN links  |APPLICATION|Cybersecurity Team|
|INC-2024-08801|P1      |DISA-WAN-DC-01 packet loss exceeding threshold    |NETWORK    |OE Network Team   |
|INC-2024-08802|P2      |Pentagon WAN router hardware degradation          |NETWORK    |DNS/DHCP Team     |
|INC-2024-08803|P2      |Devices not checking in to Intune MDM             |DEVICE     |Cybersecurity Team|
|INC-2024-08804|P2      |Infoblox primary DNS elevated latency             |NETWORK    |Cybersecurity Team|
|INC-2024-08813|P2      |MFA failures for PIV/CAC users — Pentagon building|SECURITY   |OE Network Team   |

In [16]:
# Ingestion Status
print('=== Data Source Ingestion Status ===')
spark.read.format('delta').load(Paths.gold('data_source_ingestion_status')) \
    .select('source_name','status','records_last_batch',
            'data_quality_score','latency_ms','error_message') \
    .show(truncate=False)

=== Data Source Ingestion Status ===
+------------+------+------------------+------------------+----------+--------------------------------------------+
|source_name |status|records_last_batch|data_quality_score|latency_ms|error_message                               |
+------------+------+------------------+------------------+----------+--------------------------------------------+
|Infoblox    |ACTIVE|8000              |94.8              |380       |Elevated DNS query latency on primary server|
|Salesforce  |ACTIVE|20                |98.5              |120       |NULL                                        |
|ScienceLogic|ACTIVE|1397              |98.5              |120       |NULL                                        |
|Aternity    |ACTIVE|2500              |98.5              |120       |NULL                                        |
|NetScout    |ACTIVE|3000              |98.5              |120       |NULL                                        |
|Intune      |ACTIVE|400           